<a href="https://colab.research.google.com/github/Ashwini1505-png/codealpha/blob/main/webrtcapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn python-socketio nest-asyncio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 1.9 MB/s eta 0:00:00


In [2]:
import socketio
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

# Create a Socket.io server with ASGI (async) support
sio = socketio.AsyncServer(async_mode='asgi', cors_allowed_origins='*')
app = FastAPI()

# Wrap FastAPI with Socket.io
socket_app = socketio.ASGIApp(sio, other_asgi_app=app)

# Dictionary to keep track of connected users
users = {}

@sio.on('connect')
async def connect(sid, environ):
    print(f"User connected: {sid}")

@sio.on('join')
async def join(sid, username):
    users[sid] = username
    # Tell everyone else that someone joined
    await sio.emit('user_joined', {'sid': sid, 'username': username}, skip_sid=sid)

@sio.on('disconnect')
async def disconnect(sid):
    if sid in users:
        print(f"User disconnected: {users[sid]}")
        await sio.emit('user_left', {'sid': sid, 'username': users[sid]}, skip_sid=sid)
        del users[sid]

# --- WebRTC Signaling ---
@sio.on('offer')
async def offer(sid, data):
    await sio.emit('offer', data, room=data['to'])

@sio.on('answer')
async def answer(sid, data):
    await sio.emit('answer', data, room=data['to'])

@sio.on('ice_candidate')
async def ice_candidate(sid, data):
    await sio.emit('ice_candidate', data, room=data['to'])

# --- Collaboration Tools ---
@sio.on('chat_message')
async def chat_message(sid, data):
    data['sender'] = users.get(sid, "Unknown")
    await sio.emit('chat_message', data)

@sio.on('draw')
async def draw(sid, data):
    # Broadcast drawing coordinates to everyone else
    await sio.emit('draw', data, skip_sid=sid)

print("✅ Backend Signaling Server Ready!")

✅ Backend Signaling Server Ready!


In [3]:
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Colab Meet & Collab</title>
    <script src="[https://cdn.socket.io/4.7.2/socket.io.min.js](https://cdn.socket.io/4.7.2/socket.io.min.js)"></script>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: #1e1e2f; color: white; margin: 0; padding: 20px; }
        .grid { display: grid; grid-template-columns: 2fr 1fr; gap: 20px; }
        .panel { background: #2a2a40; padding: 15px; border-radius: 10px; }
        .video-container { display: flex; gap: 10px; margin-bottom: 15px; }
        video { width: 100%; max-width: 300px; border-radius: 8px; background: #000; transform: scaleX(-1); } /* Mirror effect */
        canvas { background: white; border-radius: 8px; cursor: crosshair; width: 100%; height: 300px; }
        button { background: #4caf50; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; margin-right: 5px; }
        button:hover { background: #45a049; }
        .danger { background: #f44336; }
        .chat-box { height: 200px; overflow-y: auto; background: #1e1e2f; padding: 10px; margin-bottom: 10px; border-radius: 5px; }
        input[type="text"] { width: 70%; padding: 10px; border-radius: 5px; border: none; }
    </style>
</head>
<body>
    <div id="auth-screen">
        <h2>Enter your name to join</h2>
        <input type="text" id="username" placeholder="Your Name">
        <button onclick="joinRoom()">Join</button>
    </div>

    <div id="app-screen" style="display: none;" class="grid">
        <!-- LEFT COLUMN: Video & Whiteboard -->
        <div class="panel">
            <h3>🎥 Video Call</h3>
            <div class="video-container">
                <video id="localVideo" autoplay muted playsinline></video>
                <video id="remoteVideo" autoplay playsinline style="transform: scaleX(1);"></video>
            </div>
            <div>
                <button onclick="startVideo()">Start Camera</button>
                <button onclick="shareScreen()">Share Screen</button>
                <button onclick="callPeer()" id="callBtn" disabled>Call Peer</button>
            </div>

            <h3 style="margin-top: 20px;">🖍️ Shared Whiteboard</h3>
            <canvas id="whiteboard" width="600" height="300"></canvas>
            <button onclick="clearCanvas()" style="margin-top: 10px;">Clear</button>
        </div>

        <!-- RIGHT COLUMN: Chat & Files -->
        <div class="panel">
            <h3>💬 Chat & Files</h3>
            <div class="chat-box" id="chat"></div>
            <input type="text" id="msg" placeholder="Type a message...">
            <button onclick="sendMessage()">Send</button>

            <hr style="border-color: #444; margin: 20px 0;">

            <h4>📁 Share File</h4>
            <input type="file" id="fileInput" onchange="sendFile()">
        </div>
    </div>

    <script>
        const socket = io();
        let localStream;
        let peerConnection;
        let remoteSid = null;
        let username = "";

        const configuration = { 'iceServers': [{'urls': 'stun:stun.l.google.com:19302'}] };

        // --- 1. SETUP & AUTH ---
        function joinRoom() {
            username = document.getElementById('username').value;
            if (!username) return alert("Enter a name!");

            document.getElementById('auth-screen').style.display = 'none';
            document.getElementById('app-screen').style.display = 'grid';
            socket.emit('join', username);
        }

        socket.on('user_joined', (data) => {
            addChatMsg("System", `${data.username} joined! You can now call them.`);
            remoteSid = data.sid;
            document.getElementById('callBtn').disabled = false;
        });

        socket.on('user_left', (data) => {
            addChatMsg("System", `${data.username} left.`);
            remoteSid = null;
            document.getElementById('remoteVideo').srcObject = null;
        });

        // --- 2. WEBRTC (VIDEO & SCREEN SHARE) ---
        async function startVideo() {
            try {
                localStream = await navigator.mediaDevices.getUserMedia({ video: true, audio: true });
                document.getElementById('localVideo').srcObject = localStream;
            } catch (e) { alert("Camera access denied or unavailable."); }
        }

        async function shareScreen() {
            try {
                const screenStream = await navigator.mediaDevices.getDisplayMedia({ video: true });
                document.getElementById('localVideo').srcObject = screenStream;
                // Replace video track in peer connection if active
                if (peerConnection) {
                    const videoTrack = screenStream.getVideoTracks()[0];
                    const sender = peerConnection.getSenders().find(s => s.track.kind === videoTrack.kind);
                    sender.replaceTrack(videoTrack);
                }
            } catch (e) { console.error("Screen share failed", e); }
        }

        function createPeerConnection() {
            peerConnection = new RTCPeerConnection(configuration);

            // Add our local stream tracks to the connection
            if (localStream) {
                localStream.getTracks().forEach(track => peerConnection.addTrack(track, localStream));
            }

            // Receive remote stream
            peerConnection.ontrack = (event) => {
                document.getElementById('remoteVideo').srcObject = event.streams[0];
            };

            // Send ICE candidates to peer
            peerConnection.onicecandidate = (event) => {
                if (event.candidate && remoteSid) {
                    socket.emit('ice_candidate', { to: remoteSid, candidate: event.candidate });
                }
            };
        }

        async function callPeer() {
            if (!remoteSid) return alert("No one is online to call.");
            createPeerConnection();
            const offer = await peerConnection.createOffer();
            await peerConnection.setLocalDescription(offer);
            socket.emit('offer', { to: remoteSid, offer: offer });
            addChatMsg("System", "Calling peer...");
        }

        // Handle incoming WebRTC signaling
        socket.on('offer', async (data) => {
            remoteSid = data.from || remoteSid; // Quick hack for 1-to-1 demo
            createPeerConnection();
            await peerConnection.setRemoteDescription(new RTCSessionDescription(data.offer));
            const answer = await peerConnection.createAnswer();
            await peerConnection.setLocalDescription(answer);
            socket.emit('answer', { to: remoteSid, answer: answer });
        });

        socket.on('answer', async (data) => {
            await peerConnection.setRemoteDescription(new RTCSessionDescription(data.answer));
        });

        socket.on('ice_candidate', async (data) => {
            if (peerConnection) {
                await peerConnection.addIceCandidate(new RTCIceCandidate(data.candidate));
            }
        });

        // --- 3. WHITEBOARD ---
        const canvas = document.getElementById('whiteboard');
        const ctx = canvas.getContext('2d');
        let drawing = false;

        function getMousePos(evt) {
            const rect = canvas.getBoundingClientRect();
            // Scale coordinates based on actual display size vs internal size
            const scaleX = canvas.width / rect.width;
            const scaleY = canvas.height / rect.height;
            return {
                x: (evt.clientX - rect.left) * scaleX,
                y: (evt.clientY - rect.top) * scaleY
            };
        }

        canvas.addEventListener('mousedown', (e) => { drawing = true; drawStart(e); });
        canvas.addEventListener('mouseup', () => { drawing = false; ctx.beginPath(); });
        canvas.addEventListener('mousemove', (e) => { if (drawing) drawLine(e, true); });

        function drawStart(e) {
            const pos = getMousePos(e);
            ctx.moveTo(pos.x, pos.y);
            ctx.beginPath();
        }

        function drawLine(e, emit) {
            const pos = getMousePos(e);
            ctx.lineTo(pos.x, pos.y);
            ctx.stroke();
            if (emit) {
                socket.emit('draw', { x: pos.x, y: pos.y });
            }
        }

        socket.on('draw', (data) => {
            ctx.lineTo(data.x, data.y);
            ctx.stroke();
            ctx.beginPath();
            ctx.moveTo(data.x, data.y);
        });

        function clearCanvas() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
        }

        // --- 4. CHAT & FILE SHARE ---
        function sendMessage() {
            const msg = document.getElementById('msg').value;
            socket.emit('chat_message', { text: msg });
            document.getElementById('msg').value = '';
        }

        socket.on('chat_message', (data) => {
            addChatMsg(data.sender, data.text, data.isFile);
        });

        function addChatMsg(sender, text, isFile=false) {
            const box = document.getElementById('chat');
            const content = isFile ? `<a href="${text}" download="shared_file" style="color:#4caf50;">📁 Download File</a>` : text;
            box.innerHTML += `<p><strong>${sender}:</strong> ${content}</p>`;
            box.scrollTop = box.scrollHeight;
        }

        function sendFile() {
            const file = document.getElementById('fileInput').files[0];
            const reader = new FileReader();
            reader.onload = function(evt) {
                socket.emit('chat_message', { text: evt.target.result, isFile: true });
            };
            reader.readAsDataURL(file); // Convert file to base64 to send over socket
        }
    </script>
</body>
</html>
"""

# Serve the HTML page on the root URL
@app.get("/")
def get_home():
    return HTMLResponse(content=html_content)

print("✅ Frontend Interface Loaded!")


✅ Frontend Interface Loaded!


In [4]:
import nest_asyncio
import uvicorn
import threading
from google.colab import output

# Allow asyncio to run inside Colab's existing loop
nest_asyncio.apply()

# Define the runner
def run_app():
    # Note: We run 'socket_app' which contains BOTH FastAPI and Socket.io
    uvicorn.run(socket_app, host="127.0.0.1", port=8000)

# Start server in the background
threading.Thread(target=run_app, daemon=True).start()

print("🚀 WebRTC Server is running!")
print("👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇")
output.serve_kernel_port_as_window(8000)

🚀 WebRTC Server is running!
👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [5]:
!pip install fastapi uvicorn python-socketio nest-asyncio

In [6]:
import socketio
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

# Create a Socket.io server with ASGI (async) support
sio = socketio.AsyncServer(async_mode='asgi', cors_allowed_origins='*')
app = FastAPI()

# Wrap FastAPI with Socket.io
socket_app = socketio.ASGIApp(sio, other_asgi_app=app)

# Dictionary to keep track of connected users
users = {}

@sio.on('connect')
async def connect(sid, environ):
    print(f"User connected: {sid}")

@sio.on('join')
async def join(sid, username):
    users[sid] = username
    # Tell everyone else that someone joined
    await sio.emit('user_joined', {'sid': sid, 'username': username}, skip_sid=sid)

@sio.on('disconnect')
async def disconnect(sid):
    if sid in users:
        print(f"User disconnected: {users[sid]}")
        await sio.emit('user_left', {'sid': sid, 'username': users[sid]}, skip_sid=sid)
        del users[sid]

# --- WebRTC Signaling ---
@sio.on('offer')
async def offer(sid, data):
    await sio.emit('offer', data, room=data['to'])

@sio.on('answer')
async def answer(sid, data):
    await sio.emit('answer', data, room=data['to'])

@sio.on('ice_candidate')
async def ice_candidate(sid, data):
    await sio.emit('ice_candidate', data, room=data['to'])

# --- Collaboration Tools ---
@sio.on('chat_message')
async def chat_message(sid, data):
    data['sender'] = users.get(sid, "Unknown")
    await sio.emit('chat_message', data)

@sio.on('draw')
async def draw(sid, data):
    # Broadcast drawing coordinates to everyone else
    await sio.emit('draw', data, skip_sid=sid)

print("✅ Backend Signaling Server Ready!")

✅ Backend Signaling Server Ready!


In [7]:
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Colab Meet & Collab</title>
    <!-- EXACT URL NEEDED BELOW. NO BRACKETS. -->
    <script src="[https://cdn.socket.io/4.7.2/socket.io.min.js](https://cdn.socket.io/4.7.2/socket.io.min.js)"></script>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: #1e1e2f; color: white; margin: 0; padding: 20px; }
        .grid { display: grid; grid-template-columns: 2fr 1fr; gap: 20px; }
        .panel { background: #2a2a40; padding: 15px; border-radius: 10px; }
        .video-container { display: flex; gap: 10px; margin-bottom: 15px; }
        video { width: 100%; max-width: 300px; border-radius: 8px; background: #000; transform: scaleX(-1); } /* Mirror effect */
        canvas { background: white; border-radius: 8px; cursor: crosshair; width: 100%; height: 300px; }
        button { background: #4caf50; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; margin-right: 5px; }
        button:hover { background: #45a049; }
        button:disabled { background: #555; cursor: not-allowed; }
        .danger { background: #f44336; }
        .chat-box { height: 200px; overflow-y: auto; background: #1e1e2f; padding: 10px; margin-bottom: 10px; border-radius: 5px; }
        input[type="text"] { width: 70%; padding: 10px; border-radius: 5px; border: none; }

        #auth-screen { text-align: center; margin-top: 100px; }
        #auth-screen input { width: 300px; margin-bottom: 15px; }
    </style>
</head>
<body>
    <div id="auth-screen">
        <h2>Enter your name to join</h2>
        <input type="text" id="username" placeholder="Your Name" onkeypress="if(event.key === 'Enter') joinRoom()">
        <br>
        <button onclick="joinRoom()" style="width: 300px;">Join</button>
    </div>

    <div id="app-screen" style="display: none;" class="grid">
        <!-- LEFT COLUMN: Video & Whiteboard -->
        <div class="panel">
            <h3>🎥 Video Call</h3>
            <div class="video-container">
                <video id="localVideo" autoplay muted playsinline></video>
                <video id="remoteVideo" autoplay playsinline style="transform: scaleX(1);"></video>
            </div>
            <div>
                <button onclick="startVideo()">Start Camera</button>
                <button onclick="shareScreen()">Share Screen</button>
                <button onclick="callPeer()" id="callBtn" disabled>Call Peer</button>
            </div>

            <h3 style="margin-top: 20px;">🖍️ Shared Whiteboard</h3>
            <canvas id="whiteboard" width="600" height="300"></canvas>
            <button onclick="clearCanvas()" style="margin-top: 10px;" class="danger">Clear</button>
        </div>

        <!-- RIGHT COLUMN: Chat & Files -->
        <div class="panel">
            <h3>💬 Chat & Files</h3>
            <div class="chat-box" id="chat"></div>
            <input type="text" id="msg" placeholder="Type a message..." onkeypress="if(event.key === 'Enter') sendMessage()">
            <button onclick="sendMessage()">Send</button>

            <hr style="border-color: #444; margin: 20px 0;">

            <h4>📁 Share File</h4>
            <input type="file" id="fileInput" onchange="sendFile()">
        </div>
    </div>

    <script>
        // Use default Socket.io connection which adapts to Colab's port forwarding
        const socket = io();
        let localStream;
        let peerConnection;
        let remoteSid = null;
        let username = "";

        const configuration = { 'iceServers': [{'urls': 'stun:stun.l.google.com:19302'}] };

        // --- 1. SETUP & AUTH ---
        function joinRoom() {
            username = document.getElementById('username').value.trim();
            if (!username) return alert("Enter a name!");

            document.getElementById('auth-screen').style.display = 'none';
            document.getElementById('app-screen').style.display = 'grid';
            socket.emit('join', username);
            addChatMsg("System", "You joined the room! Waiting for others...");
        }

        socket.on('user_joined', (data) => {
            addChatMsg("System", `${data.username} joined! You can now call them.`);
            remoteSid = data.sid;
            document.getElementById('callBtn').disabled = false;
        });

        socket.on('user_left', (data) => {
            addChatMsg("System", `${data.username} left.`);
            remoteSid = null;
            document.getElementById('remoteVideo').srcObject = null;
            document.getElementById('callBtn').disabled = true;
        });

        // --- 2. WEBRTC (VIDEO & SCREEN SHARE) ---
        async function startVideo() {
            try {
                localStream = await navigator.mediaDevices.getUserMedia({ video: true, audio: true });
                document.getElementById('localVideo').srcObject = localStream;
            } catch (e) { alert("Camera access denied or unavailable."); }
        }

        async function shareScreen() {
            try {
                const screenStream = await navigator.mediaDevices.getDisplayMedia({ video: true });
                document.getElementById('localVideo').srcObject = screenStream;
                document.getElementById('localVideo').style.transform = "scaleX(1)"; // Un-mirror for screen share
                if (peerConnection) {
                    const videoTrack = screenStream.getVideoTracks()[0];
                    const sender = peerConnection.getSenders().find(s => s.track.kind === videoTrack.kind);
                    if(sender) sender.replaceTrack(videoTrack);
                }
            } catch (e) { console.error("Screen share failed", e); }
        }

        function createPeerConnection() {
            peerConnection = new RTCPeerConnection(configuration);

            if (localStream) {
                localStream.getTracks().forEach(track => peerConnection.addTrack(track, localStream));
            }

            peerConnection.ontrack = (event) => {
                document.getElementById('remoteVideo').srcObject = event.streams[0];
            };

            peerConnection.onicecandidate = (event) => {
                if (event.candidate && remoteSid) {
                    socket.emit('ice_candidate', { to: remoteSid, candidate: event.candidate });
                }
            };
        }

        async function callPeer() {
            if (!remoteSid) return alert("No one is online to call.");
            createPeerConnection();
            const offer = await peerConnection.createOffer();
            await peerConnection.setLocalDescription(offer);
            socket.emit('offer', { to: remoteSid, offer: offer });
            addChatMsg("System", "Calling peer...");
        }

        socket.on('offer', async (data) => {
            remoteSid = data.from || remoteSid; // Hack for 1-to-1
            createPeerConnection();
            await peerConnection.setRemoteDescription(new RTCSessionDescription(data.offer));
            const answer = await peerConnection.createAnswer();
            await peerConnection.setLocalDescription(answer);
            socket.emit('answer', { to: remoteSid, answer: answer });
        });

        socket.on('answer', async (data) => {
            await peerConnection.setRemoteDescription(new RTCSessionDescription(data.answer));
        });

        socket.on('ice_candidate', async (data) => {
            if (peerConnection) {
                await peerConnection.addIceCandidate(new RTCIceCandidate(data.candidate));
            }
        });

        // --- 3. WHITEBOARD ---
        const canvas = document.getElementById('whiteboard');
        const ctx = canvas.getContext('2d');
        let drawing = false;

        function getMousePos(evt) {
            const rect = canvas.getBoundingClientRect();
            const scaleX = canvas.width / rect.width;
            const scaleY = canvas.height / rect.height;
            return {
                x: (evt.clientX - rect.left) * scaleX,
                y: (evt.clientY - rect.top) * scaleY
            };
        }

        canvas.addEventListener('mousedown', (e) => { drawing = true; drawStart(e); });
        canvas.addEventListener('mouseup', () => { drawing = false; ctx.beginPath(); });
        canvas.addEventListener('mouseout', () => { drawing = false; ctx.beginPath(); });
        canvas.addEventListener('mousemove', (e) => { if (drawing) drawLine(e, true); });

        function drawStart(e) {
            const pos = getMousePos(e);
            ctx.moveTo(pos.x, pos.y);
            ctx.beginPath();
        }

        function drawLine(e, emit) {
            const pos = getMousePos(e);
            ctx.lineTo(pos.x, pos.y);
            ctx.stroke();
            if (emit) {
                socket.emit('draw', { x: pos.x, y: pos.y });
            }
        }

        socket.on('draw', (data) => {
            ctx.lineTo(data.x, data.y);
            ctx.stroke();
            ctx.beginPath();
            ctx.moveTo(data.x, data.y);
        });

        function clearCanvas() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
        }

        // --- 4. CHAT & FILE SHARE ---
        function sendMessage() {
            const msgInput = document.getElementById('msg');
            if (!msgInput.value.trim()) return;
            socket.emit('chat_message', { text: msgInput.value });
            msgInput.value = '';
        }

        socket.on('chat_message', (data) => {
            addChatMsg(data.sender, data.text, data.isFile);
        });

        function addChatMsg(sender, text, isFile=false) {
            const box = document.getElementById('chat');
            const content = isFile ? `<a href="${text}" download="shared_file" style="color:#4caf50;">📁 Download File</a>` : text;
            box.innerHTML += `<p><strong style="color: ${sender === username ? '#4caf50' : '#2196F3'}">${sender}:</strong> ${content}</p>`;
            box.scrollTop = box.scrollHeight;
        }

        function sendFile() {
            const file = document.getElementById('fileInput').files[0];
            if (!file) return;
            const reader = new FileReader();
            reader.onload = function(evt) {
                socket.emit('chat_message', { text: evt.target.result, isFile: true });
            };
            reader.readAsDataURL(file); // Convert file to base64 to send over socket
            document.getElementById('fileInput').value = ""; // Reset input
        }
    </script>
</body>
</html>
"""

# Serve the HTML page on the root URL
@app.get("/")
def get_home():
    return HTMLResponse(content=html_content)

print("✅ Frontend Interface Loaded!")

✅ Frontend Interface Loaded!


In [9]:
import nest_asyncio
import uvicorn
import threading
from google.colab import output

# Allow asyncio to run inside Colab's existing loop
nest_asyncio.apply()

# Define the runner
def run_app():
    # Note: We run 'socket_app' which contains BOTH FastAPI and Socket.io
    uvicorn.run(socket_app, host="127.0.0.1", port=8000)

# Start server in the background
threading.Thread(target=run_app, daemon=True).start()

print("🚀 WebRTC Server is running!")
print("👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇")
output.serve_kernel_port_as_window(8000)

🚀 WebRTC Server is running!
👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

INFO:     Started server process [787]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('127.0.0.1', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [10]:
import nest_asyncio
import uvicorn
import threading
import portpicker
from google.colab import output

# Allow asyncio to run inside Colab's existing loop
nest_asyncio.apply()

# Pick an unused port to avoid "address already in use" errors!
port = portpicker.pick_unused_port()

# Define the runner
def run_app():
    # Note: We run 'socket_app' which contains BOTH FastAPI and Socket.io
    uvicorn.run(socket_app, host="127.0.0.1", port=port)

# Start server in the background
threading.Thread(target=run_app, daemon=True).start()

print(f"🚀 WebRTC Server is running on port {port}!")
print("👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇")
print("(Note: You can safely ignore the warning about 'browser security' or 'iframe')")
output.serve_kernel_port_as_window(port)


🚀 WebRTC Server is running on port 40301!
👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇
(Note: You can safely ignore the warning about 'browser security' or 'iframe')
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

INFO:     Started server process [787]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:40301 (Press CTRL+C to quit)


In [11]:
!pip install fastapi uvicorn python-socketio nest-asyncio

In [12]:
import socketio
from fastapi import FastAPI
from fastapi.responses import HTMLResponse

# Create a Socket.io server with ASGI (async) support
sio = socketio.AsyncServer(async_mode='asgi', cors_allowed_origins='*')
app = FastAPI()

# Wrap FastAPI with Socket.io
socket_app = socketio.ASGIApp(sio, other_asgi_app=app)

# Dictionary to keep track of connected users
users = {}

@sio.on('connect')
async def connect(sid, environ):
    print(f"User connected: {sid}")

@sio.on('join')
async def join(sid, username):
    users[sid] = username
    # Tell everyone else that someone joined
    await sio.emit('user_joined', {'sid': sid, 'username': username}, skip_sid=sid)

@sio.on('disconnect')
async def disconnect(sid):
    if sid in users:
        print(f"User disconnected: {users[sid]}")
        await sio.emit('user_left', {'sid': sid, 'username': users[sid]}, skip_sid=sid)
        del users[sid]

# --- WebRTC Signaling ---
@sio.on('offer')
async def offer(sid, data):
    await sio.emit('offer', data, room=data['to'])

@sio.on('answer')
async def answer(sid, data):
    await sio.emit('answer', data, room=data['to'])

@sio.on('ice_candidate')
async def ice_candidate(sid, data):
    await sio.emit('ice_candidate', data, room=data['to'])

# --- Collaboration Tools ---
@sio.on('chat_message')
async def chat_message(sid, data):
    data['sender'] = users.get(sid, "Unknown")
    await sio.emit('chat_message', data)

@sio.on('draw')
async def draw(sid, data):
    # Broadcast drawing coordinates to everyone else
    await sio.emit('draw', data, skip_sid=sid)

print("✅ Backend Signaling Server Ready!")

✅ Backend Signaling Server Ready!


In [13]:
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Colab Meet & Collab</title>
    <!-- We use a protocol-relative URL to prevent copy/paste auto-linkers from breaking the code! -->
    <script src="//cdn.socket.io/4.7.2/socket.io.min.js"></script>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: #1e1e2f; color: white; margin: 0; padding: 20px; }
        .grid { display: grid; grid-template-columns: 2fr 1fr; gap: 20px; }
        .panel { background: #2a2a40; padding: 15px; border-radius: 10px; }
        .video-container { display: flex; gap: 10px; margin-bottom: 15px; }
        video { width: 100%; max-width: 300px; border-radius: 8px; background: #000; transform: scaleX(-1); } /* Mirror effect */
        canvas { background: white; border-radius: 8px; cursor: crosshair; width: 100%; height: 300px; }
        button { background: #4caf50; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; margin-right: 5px; }
        button:hover { background: #45a049; }
        button:disabled { background: #555; cursor: not-allowed; }
        .danger { background: #f44336; }
        .chat-box { height: 200px; overflow-y: auto; background: #1e1e2f; padding: 10px; margin-bottom: 10px; border-radius: 5px; }
        input[type="text"] { width: 70%; padding: 10px; border-radius: 5px; border: none; }

        #auth-screen { text-align: center; margin-top: 100px; }
        #auth-screen input { width: 300px; margin-bottom: 15px; }
    </style>
</head>
<body>
    <div id="auth-screen">
        <h2>Enter your name to join</h2>
        <input type="text" id="username" placeholder="Your Name" onkeypress="if(event.key === 'Enter') joinRoom()">
        <br>
        <button onclick="joinRoom()" style="width: 300px;">Join</button>
    </div>

    <div id="app-screen" style="display: none;" class="grid">
        <!-- LEFT COLUMN: Video & Whiteboard -->
        <div class="panel">
            <h3>🎥 Video Call</h3>
            <div class="video-container">
                <video id="localVideo" autoplay muted playsinline></video>
                <video id="remoteVideo" autoplay playsinline style="transform: scaleX(1);"></video>
            </div>
            <div>
                <button onclick="startVideo()">Start Camera</button>
                <button onclick="shareScreen()">Share Screen</button>
                <button onclick="callPeer()" id="callBtn" disabled>Call Peer</button>
            </div>

            <h3 style="margin-top: 20px;">🖍️ Shared Whiteboard</h3>
            <canvas id="whiteboard" width="600" height="300"></canvas>
            <button onclick="clearCanvas()" style="margin-top: 10px;" class="danger">Clear</button>
        </div>

        <!-- RIGHT COLUMN: Chat & Files -->
        <div class="panel">
            <h3>💬 Chat & Files</h3>
            <div class="chat-box" id="chat"></div>
            <input type="text" id="msg" placeholder="Type a message..." onkeypress="if(event.key === 'Enter') sendMessage()">
            <button onclick="sendMessage()">Send</button>

            <hr style="border-color: #444; margin: 20px 0;">

            <h4>📁 Share File</h4>
            <input type="file" id="fileInput" onchange="sendFile()">
        </div>
    </div>

    <script>
        // Force WebSockets for better compatibility with Google Colab's network proxy
        const socket = io({ transports: ['websocket'] });
        let localStream;
        let peerConnection;
        let remoteSid = null;
        let username = "";

        const configuration = { 'iceServers': [{'urls': 'stun:stun.l.google.com:19302'}] };

        // --- 1. SETUP & AUTH ---
        function joinRoom() {
            username = document.getElementById('username').value.trim();
            if (!username) return alert("Enter a name!");

            document.getElementById('auth-screen').style.display = 'none';
            document.getElementById('app-screen').style.display = 'grid';
            socket.emit('join', username);
            addChatMsg("System", "You joined the room! Waiting for others...");
        }

        socket.on('user_joined', (data) => {
            addChatMsg("System", `${data.username} joined! You can now call them.`);
            remoteSid = data.sid;
            document.getElementById('callBtn').disabled = false;
        });

        socket.on('user_left', (data) => {
            addChatMsg("System", `${data.username} left.`);
            remoteSid = null;
            document.getElementById('remoteVideo').srcObject = null;
            document.getElementById('callBtn').disabled = true;
        });

        // --- 2. WEBRTC (VIDEO & SCREEN SHARE) ---
        async function startVideo() {
            try {
                localStream = await navigator.mediaDevices.getUserMedia({ video: true, audio: true });
                document.getElementById('localVideo').srcObject = localStream;
            } catch (e) { alert("Camera access denied or unavailable."); }
        }

        async function shareScreen() {
            try {
                const screenStream = await navigator.mediaDevices.getDisplayMedia({ video: true });
                document.getElementById('localVideo').srcObject = screenStream;
                document.getElementById('localVideo').style.transform = "scaleX(1)"; // Un-mirror for screen share
                if (peerConnection) {
                    const videoTrack = screenStream.getVideoTracks()[0];
                    const sender = peerConnection.getSenders().find(s => s.track.kind === videoTrack.kind);
                    if(sender) sender.replaceTrack(videoTrack);
                }
            } catch (e) { console.error("Screen share failed", e); }
        }

        function createPeerConnection() {
            peerConnection = new RTCPeerConnection(configuration);

            if (localStream) {
                localStream.getTracks().forEach(track => peerConnection.addTrack(track, localStream));
            }

            peerConnection.ontrack = (event) => {
                document.getElementById('remoteVideo').srcObject = event.streams[0];
            };

            peerConnection.onicecandidate = (event) => {
                if (event.candidate && remoteSid) {
                    socket.emit('ice_candidate', { to: remoteSid, candidate: event.candidate });
                }
            };
        }

        async function callPeer() {
            if (!remoteSid) return alert("No one is online to call.");
            createPeerConnection();
            const offer = await peerConnection.createOffer();
            await peerConnection.setLocalDescription(offer);
            socket.emit('offer', { to: remoteSid, offer: offer });
            addChatMsg("System", "Calling peer...");
        }

        socket.on('offer', async (data) => {
            remoteSid = data.from || remoteSid; // Hack for 1-to-1
            createPeerConnection();
            await peerConnection.setRemoteDescription(new RTCSessionDescription(data.offer));
            const answer = await peerConnection.createAnswer();
            await peerConnection.setLocalDescription(answer);
            socket.emit('answer', { to: remoteSid, answer: answer });
        });

        socket.on('answer', async (data) => {
            await peerConnection.setRemoteDescription(new RTCSessionDescription(data.answer));
        });

        socket.on('ice_candidate', async (data) => {
            if (peerConnection) {
                await peerConnection.addIceCandidate(new RTCIceCandidate(data.candidate));
            }
        });

        // --- 3. WHITEBOARD ---
        const canvas = document.getElementById('whiteboard');
        const ctx = canvas.getContext('2d');
        let drawing = false;

        function getMousePos(evt) {
            const rect = canvas.getBoundingClientRect();
            const scaleX = canvas.width / rect.width;
            const scaleY = canvas.height / rect.height;
            return {
                x: (evt.clientX - rect.left) * scaleX,
                y: (evt.clientY - rect.top) * scaleY
            };
        }

        canvas.addEventListener('mousedown', (e) => { drawing = true; drawStart(e); });
        canvas.addEventListener('mouseup', () => { drawing = false; ctx.beginPath(); });
        canvas.addEventListener('mouseout', () => { drawing = false; ctx.beginPath(); });
        canvas.addEventListener('mousemove', (e) => { if (drawing) drawLine(e, true); });

        function drawStart(e) {
            const pos = getMousePos(e);
            ctx.moveTo(pos.x, pos.y);
            ctx.beginPath();
        }

        function drawLine(e, emit) {
            const pos = getMousePos(e);
            ctx.lineTo(pos.x, pos.y);
            ctx.stroke();
            if (emit) {
                socket.emit('draw', { x: pos.x, y: pos.y });
            }
        }

        socket.on('draw', (data) => {
            ctx.lineTo(data.x, data.y);
            ctx.stroke();
            ctx.beginPath();
            ctx.moveTo(data.x, data.y);
        });

        function clearCanvas() {
            ctx.clearRect(0, 0, canvas.width, canvas.height);
        }

        // --- 4. CHAT & FILE SHARE ---
        function sendMessage() {
            const msgInput = document.getElementById('msg');
            if (!msgInput.value.trim()) return;
            socket.emit('chat_message', { text: msgInput.value });
            msgInput.value = '';
        }

        socket.on('chat_message', (data) => {
            addChatMsg(data.sender, data.text, data.isFile);
        });

        function addChatMsg(sender, text, isFile=false) {
            const box = document.getElementById('chat');
            const content = isFile ? `<a href="${text}" download="shared_file" style="color:#4caf50;">📁 Download File</a>` : text;
            box.innerHTML += `<p><strong style="color: ${sender === username ? '#4caf50' : '#2196F3'}">${sender}:</strong> ${content}</p>`;
            box.scrollTop = box.scrollHeight;
        }

        function sendFile() {
            const file = document.getElementById('fileInput').files[0];
            if (!file) return;
            const reader = new FileReader();
            reader.onload = function(evt) {
                socket.emit('chat_message', { text: evt.target.result, isFile: true });
            };
            reader.readAsDataURL(file); // Convert file to base64 to send over socket
            document.getElementById('fileInput').value = ""; // Reset input
        }
    </script>
</body>
</html>
"""

# Serve the HTML page on the root URL
@app.get("/")
def get_home():
    return HTMLResponse(content=html_content)

print("✅ Frontend Interface Loaded!")

✅ Frontend Interface Loaded!


In [14]:
import nest_asyncio
import uvicorn
import threading
import portpicker
from google.colab import output

# Allow asyncio to run inside Colab's existing loop
nest_asyncio.apply()

# Pick an unused port to avoid "address already in use" errors!
port = portpicker.pick_unused_port()

# Define the runner
def run_app():
    # Note: We run 'socket_app' which contains BOTH FastAPI and Socket.io
    uvicorn.run(socket_app, host="127.0.0.1", port=port)

# Start server in the background
threading.Thread(target=run_app, daemon=True).start()

print(f"🚀 WebRTC Server is running on port {port}!")
print("👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇")
print("(Note: You can safely ignore the warning about 'browser security' or 'iframe')")
output.serve_kernel_port_as_window(port)


🚀 WebRTC Server is running on port 43867!
👇 CLICK THE LINK BELOW TO OPEN YOUR APP 👇
(Note: You can safely ignore the warning about 'browser security' or 'iframe')
Try `serve_kernel_port_as_iframe` instead. 


INFO:     Started server process [787]


<IPython.core.display.Javascript object>

INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:43867 (Press CTRL+C to quit)
